# Comparación de modelos ML — predicciones diarias

Entrena y evalúa cuatro modelos de regresión (Ridge, Random Forest, Gradient Boosting y Regresión Lineal) sobre todos los ETFs válidos a escala diaria. Se prueba cada modelo con cinco configuraciones de variables de entrada: sin retornos pasados (0 lags) y con 1, 2, 3 y 4 días de histórico adicional.

La validación es walk-forward: el modelo aprende con los datos disponibles hasta la fecha actual y predice el retorno del día siguiente, sin ver el futuro. Los resultados de error (RMSE y MAE) se guardan en Google Sheets para su análisis posterior.

In [1]:
# pip install gspread google-auth pandas

In [2]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
import gspread

from google.oauth2.service_account import Credentials
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit


## Conexión con Google Sheets

Las funciones siguientes conectan con el sheet de experimentos, leen los bloques pendientes y guardan los resultados. Cada fila del sheet representa un par (archivo, modelo, target) con su estado: PENDIENTE, RUNNING, DONE o ERROR.

In [3]:
# CONFIGURACIÓN
DATA_DIR = "../../Datos_csv"
SERVICE_ACCOUNT_FILE = "../Modelos_semanales/credenciales_google.json"

SHEET_URL = "https://docs.google.com/spreadsheets/d/1Qc1Q1OIeSt1rMsGpo_xUQ90fhOG7RDeflyPn-cwVps0/edit?gid=1153965863#gid=1153965863"  
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]


# CONEXIÓN GOOGLE SHEETS
def conectar_sheet():
    creds = Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE,
        scopes=SCOPES
    )
    client = gspread.authorize(creds)
    spreadsheet = client.open_by_url(SHEET_URL)
    worksheet = spreadsheet.get_worksheet(0)
    return worksheet


def leer_experimentos(worksheet):
    records = worksheet.get_all_records()
    df = pd.DataFrame(records)
    return df


def buscar_primer_bloque_daily(df, metodo_buscado):
    """Igual que buscar_primer_bloque_weekly pero filtra archivos df_diario."""
    for idx, row in df.iterrows():
        estado = str(row.get("Estado", "")).strip().upper()
        archivo = str(row.get("Archivo", ""))
        metodo = str(row.get("Método", ""))

        if estado != "PENDIENTE":
            continue
        if metodo != metodo_buscado:
            continue
        if not archivo.startswith("df_diario"):  
            continue

        fila_sheet = idx + 2
        return fila_sheet, row.to_dict(), idx

    return None, None, None

In [4]:
def get_model(nombre_metodo, alpha=1.0, params=None):
    if nombre_metodo == "LinearRegression":
        return LinearRegression()
    elif nombre_metodo == "Ridge":
        return Ridge(alpha=alpha)
    elif nombre_metodo == "RandomForestRegressor":
        return RandomForestRegressor(n_jobs=1, **(params or {}))
    elif nombre_metodo == "GradientBoostingRegressor":
        return GradientBoostingRegressor(**(params or {}))
    else:
        raise ValueError(f"Método no reconocido: {nombre_metodo}")

In [5]:
def encontrar_alpha_optimo(X_train, y_train):
    alphas = [100, 500, 1000, 2000, 3000, 5000, 10000, 15000, 25000, 50000, 100000]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    tscv = TimeSeriesSplit(n_splits=5)
    ridge_cv = RidgeCV(alphas=alphas, cv=tscv)
    ridge_cv.fit(X_train_scaled, y_train)
    print(f"  Alpha óptimo encontrado: {ridge_cv.alpha_}")
    return ridge_cv.alpha_


def encontrar_params_rf(X_train, y_train):
    param_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7, 10, None],
        "min_samples_leaf": [5, 10, 20, 30],
        "max_features": ["sqrt", "log2", 0.3, 0.5]
    }
    tscv = TimeSeriesSplit(n_splits=5)
    rf = RandomForestRegressor(n_jobs=1)
    search = RandomizedSearchCV(
        estimator=rf, param_distributions=param_grid,
        n_iter=30, cv=tscv, scoring="neg_mean_squared_error", n_jobs=1
    )
    search.fit(X_train, y_train)
    print(f"  Mejores parámetros RF: {search.best_params_}")
    return search.best_params_


def encontrar_params_gbr(X_train, y_train):
    param_grid = {
        "n_estimators":     [100, 200, 300, 500],
        "learning_rate":    [0.01, 0.05, 0.1, 0.2],
        "max_depth":        [2, 3, 4, 5],
        "min_samples_leaf": [5, 10, 20, 30],
        "subsample":        [0.6, 0.8, 1.0],
        "max_features":     ["sqrt", "log2", 0.3, 0.5]
    }
    tscv = TimeSeriesSplit(n_splits=5)
    gbr = GradientBoostingRegressor()
    search = RandomizedSearchCV(
        estimator=gbr, param_distributions=param_grid,
        n_iter=30, cv=tscv, scoring="neg_mean_squared_error", n_jobs=1
    )
    search.fit(X_train, y_train)
    print(f"  Mejores parámetros GBR: {search.best_params_}")
    return search.best_params_

In [6]:
def obtener_filas_bloque_pendientes(df_exp, archivo, metodo, idx_inicio):
    filas = []
    for idx in range(idx_inicio, len(df_exp)):
        row = df_exp.iloc[idx]
        archivo_row = row["Archivo"]
        metodo_row  = row["Método"]
        estado_row  = str(row.get("Estado", "")).strip().upper()

        if archivo_row != archivo or metodo_row != metodo:
            break
        if estado_row == "PENDIENTE":
            filas.append({
                "idx_df":     idx,
                "fila_sheet": idx + 2,
                "target":     row["Target"]
            })
    return filas

In [7]:
def marcar_bloque_como_running(worksheet, filas_bloque):
    fecha_ini = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    fila_ini = filas_bloque[0]["fila_sheet"]
    fila_fin = filas_bloque[-1]["fila_sheet"]
    values = []
    for _ in filas_bloque:
        values.append([
            fecha_ini, "", "", "", "", "", "", "", "RUNNING",
            "", "", "", "", "", "", "", "", "", "", ""
        ])
    worksheet.update(range_name=f"D{fila_ini}:W{fila_fin}", values=values)
    return fecha_ini


def marcar_bloque_como_done(worksheet, filas_bloque, resultados_metricas):
    fecha_fin = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    fila_ini     = filas_bloque[0]["fila_sheet"]
    fila_fin_idx = filas_bloque[-1]["fila_sheet"]

    values = []
    for res in resultados_metricas:
        def safe(v):
            if v == "" or v is None:
                return ""
            try:
                if v != v:
                    return ""
                return float(v) if isinstance(v, (int, float)) else str(v)
            except:
                return str(v)

        values.append([
            fecha_fin,
            round(float(res["rmse"]), 5),
            round(float(res["rmse_baseline"]), 5),
            round(float(res["rmse_zeros"]), 5),
            round(float(res["mae"]), 5),
            round(float(res["mae_baseline"]), 5),
            round(float(res["mae_zeros"]), 5),
            "DONE",
            safe(res.get("alpha", "")),
            safe(res.get("n_estimators", "")),
            safe(res.get("max_depth", "")),
            safe(res.get("min_samples_leaf", "")),
            safe(res.get("max_features", "")),
            safe(res.get("learning_rate", "")),
            safe(res.get("subsample", "")),
            safe(res.get("FechaPredIni", "")),
            safe(res.get("FechaPredFin", "")),
            safe(res.get("FechaTrainIni", "")),
            safe(res.get("FechaTrainFin", ""))
        ])

    worksheet.update(range_name=f"E{fila_ini}:W{fila_fin_idx}", values=values)
    return fecha_fin


def marcar_bloque_como_error(worksheet, filas_bloque):
    fecha_fin = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    fila_ini = filas_bloque[0]["fila_sheet"]
    fila_fin = filas_bloque[-1]["fila_sheet"]
    values = []
    for _ in filas_bloque:
        values.append([
            fecha_fin, "", "", "", "", "", "", "ERROR",
            "", "", "", "", "", "", "",
            "", "", "", ""
        ])
    worksheet.update(range_name=f"E{fila_ini}:W{fila_fin}", values=values)
    return fecha_fin

In [8]:
def preprocess_daily_data(df, target_actual):
    df = df.copy()

    if target_actual not in df.columns:
        raise ValueError(f"El target '{target_actual}' no existe en el dataframe.")

    # Date viene como índice con nombre "Date"
    date_key = pd.Series(df.index.astype(str), name="Date").reset_index(drop=True)
    df = df.reset_index(drop=True)

    y = df[target_actual].copy()

    target_cols   = [c for c in df.columns if c.startswith("target_")]
    calendar_cols = [c for c in df.columns if c in (
        "fin_mes", "inicio_mes", "inicio_trimestre", "fin_trimestre",
        "día_0", "día_1", "día_2", "día_3", "día_4"
    )]

    X = df.drop(columns=target_cols + calendar_cols, errors="ignore")
    X = X.select_dtypes(include=[np.number])

    mask_valida = pd.concat([X, y.rename(target_actual)], axis=1).notna().all(axis=1)
    X        = X.loc[mask_valida].reset_index(drop=True)
    y        = y.loc[mask_valida].reset_index(drop=True)
    date_key = date_key.loc[mask_valida].reset_index(drop=True)

    return X, y, date_key

In [9]:
def temporal_split_por_date(X, y, date_key, train_fin, test_ini):
    """
    Versión diaria de temporal_split_por_weekkey.
    train_fin / test_ini son strings tipo '2024-07-19'.
    """
    date_key = date_key.astype(str)

    mask_train = date_key <= train_fin
    mask_test  = date_key >= test_ini

    X_train = X.loc[mask_train].copy()
    X_test  = X.loc[mask_test].copy()
    y_train = y.loc[mask_train].copy()
    y_test  = y.loc[mask_test].copy()

    date_key_train = date_key.loc[mask_train].copy()
    date_key_test  = date_key.loc[mask_test].copy()

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError(
            f"Split vacío con train_fin={train_fin} y test_ini={test_ini}"
        )

    info_split = {
        "n_total":      len(X),
        "n_train":      len(X_train),
        "n_test":       len(X_test),
        "train_date_ini": str(date_key_train.iloc[0]),
        "train_date_fin": str(date_key_train.iloc[-1]),
        "test_date_ini":  str(date_key_test.iloc[0]),
        "test_date_fin":  str(date_key_test.iloc[-1]),
        "pred_date_ini":  str(date_key_test.iloc[0]),
        "pred_date_fin":  str(date_key_test.iloc[-1]),
    }

    return X_train, X_test, y_train, y_test, info_split

In [10]:
# walk_forward_regression es idéntico al semanal, no cambia nada
def walk_forward_regression(X_train, y_train, X_test, y_test,
                             metodo, alpha_fijo=1.0, params_fijos=None):
    preds_model    = []
    preds_baseline = []
    preds_zeros    = []

    X_hist = X_train.copy().reset_index(drop=True)
    y_hist = y_train.copy().reset_index(drop=True)
    X_test = X_test.copy().reset_index(drop=True)
    y_test = y_test.copy().reset_index(drop=True)

    usar_scaler = metodo in {"Ridge"}

    for i in range(len(X_test)):
        if usar_scaler:
            scaler = StandardScaler()
            X_hist_fit  = scaler.fit_transform(X_hist)
            x_next_fit  = scaler.transform(X_test.iloc[[i]])
        else:
            X_hist_fit = X_hist.values
            x_next_fit = X_test.iloc[[i]].values

        model = get_model(metodo, alpha=alpha_fijo, params=params_fijos)
        model.fit(X_hist_fit, y_hist)

        preds_model.append(model.predict(x_next_fit)[0])
        preds_baseline.append(y_hist.iloc[-1])
        preds_zeros.append(0.0)

        X_hist = pd.concat([X_hist, X_test.iloc[[i]]], ignore_index=True)
        y_hist = pd.concat([y_hist, pd.Series([y_test.iloc[i]])], ignore_index=True)

    rmse          = np.sqrt(mean_squared_error(y_test, preds_model))
    rmse_baseline = np.sqrt(mean_squared_error(y_test, preds_baseline))
    rmse_zeros    = np.sqrt(mean_squared_error(y_test, preds_zeros))
    mae           = mean_absolute_error(y_test, preds_model)
    mae_baseline  = mean_absolute_error(y_test, preds_baseline)
    mae_zeros     = mean_absolute_error(y_test, preds_zeros)

    resultados = pd.DataFrame({
        "real":          y_test,
        "pred_model":    preds_model,
        "pred_baseline": preds_baseline,
        "pred_zeros":    preds_zeros
    })

    return (
        rmse, rmse_baseline, rmse_zeros,
        mae, mae_baseline, mae_zeros,
        resultados,
        alpha_fijo
    )

## Flujo de ejecución

`run_block_daily` coordina el experimento: lee el siguiente bloque pendiente, busca los mejores parámetros con los datos de train del primer target y lanza el walk-forward sobre todos los targets del bloque. Al terminar, escribe los resultados en el sheet.

In [11]:
def run_block_daily(metodo):
    worksheet = conectar_sheet()
    df_exp = leer_experimentos(worksheet)

    fila_sheet, experimento, idx_inicio = buscar_primer_bloque_daily(df_exp, metodo)

    if fila_sheet is None:
        print(f"No quedan bloques diarios pendientes para {metodo}.")
        return None

    archivo = experimento["Archivo"]

    print("=====================================================")
    print("INICIO DE BLOQUE")
    print(f"Archivo: {archivo}")
    print(f"Método: {metodo}")

    path = os.path.join(DATA_DIR, archivo)
    if not os.path.exists(path):
        raise FileNotFoundError(f"No existe el archivo: {path}")

    df_cargado = pd.read_csv(path, index_col="Date")  # ← limpio, Date como índice con nombre

    filas_bloque = obtener_filas_bloque_pendientes(df_exp, archivo, metodo, idx_inicio)

    if not filas_bloque:
        print("No hay filas pendientes válidas en el bloque.")
        return None

    target_inicial = filas_bloque[0]["target"]
    X0, y0, date_key0 = preprocess_daily_data(df_cargado, target_inicial)
    X_train0, _, y_train0, _, info_split = temporal_split_por_date(
        X=X0, y=y0, date_key=date_key0,
        train_fin=FECHA_TRAIN_FIN, test_ini=FECHA_TEST_INI
    )

    alpha_optimo   = None
    params_optimos = None

    if metodo == "Ridge":
        print("Buscando alpha óptimo para este archivo...")
        alpha_optimo = encontrar_alpha_optimo(X_train0, y_train0)
        print(f"Alpha fijado para '{archivo}': {alpha_optimo}")

    elif metodo == "RandomForestRegressor":
        print("Buscando hiperparámetros óptimos RF para este archivo...")
        params_optimos = encontrar_params_rf(X_train0, y_train0)
        print(f"Parámetros fijados para '{archivo}': {params_optimos}")

    elif metodo == "GradientBoostingRegressor":
        print("Buscando hiperparámetros óptimos GBR para este archivo...")
        params_optimos = encontrar_params_gbr(X_train0, y_train0)
        print(f"Parámetros fijados para '{archivo}': {params_optimos}")

    print("INFORMACIÓN DEL SPLIT TEMPORAL DEL BLOQUE")
    print(f"Total observaciones: {info_split['n_total']}")
    print(f"Train: {info_split['n_train']} filas")
    print(f"Test : {info_split['n_test']} filas")
    print(f"Train fechas: {info_split['train_date_ini']} -> {info_split['train_date_fin']}")
    print(f"Test  fechas: {info_split['test_date_ini']}  -> {info_split['test_date_fin']}")

    fecha_ini = marcar_bloque_como_running(worksheet, filas_bloque)

    resultados_bloque   = {}
    resultados_metricas = []

    try:
        for fila in filas_bloque:
            target = fila["target"]

            X, y, date_key = preprocess_daily_data(df_cargado, target)
            X_train, X_test, y_train, y_test, info_split_target = temporal_split_por_date(
                X=X, y=y, date_key=date_key,
                train_fin=FECHA_TRAIN_FIN, test_ini=FECHA_TEST_INI
            )

            (
                rmse, rmse_baseline, rmse_zeros,
                mae, mae_baseline, mae_zeros,
                resultados,
                alpha_usado
            ) = walk_forward_regression(
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                metodo=metodo,
                alpha_fijo=alpha_optimo if alpha_optimo is not None else 1.0,
                params_fijos=params_optimos
            )

            resultados_bloque[target] = resultados
            resultados_metricas.append({
                "rmse":             rmse,
                "rmse_baseline":    rmse_baseline,
                "rmse_zeros":       rmse_zeros,
                "mae":              mae,
                "mae_baseline":     mae_baseline,
                "mae_zeros":        mae_zeros,
                "alpha":            alpha_optimo if alpha_optimo is not None else "",
                "n_estimators":     params_optimos.get("n_estimators", "")     if params_optimos else "",
                "max_depth":        params_optimos.get("max_depth", "")        if params_optimos else "",
                "min_samples_leaf": params_optimos.get("min_samples_leaf", "") if params_optimos else "",
                "max_features":     params_optimos.get("max_features", "")     if params_optimos else "",
                "learning_rate":    params_optimos.get("learning_rate", "")    if params_optimos else "",
                "subsample":        params_optimos.get("subsample", "")        if params_optimos else "",
                "FechaTrainIni": info_split_target["train_date_ini"],
                "FechaTrainFin": info_split_target["train_date_fin"],
                "FechaPredIni":  info_split_target["pred_date_ini"],
                "FechaPredFin":  info_split_target["pred_date_fin"],
            })

            print(
                f"Target: {target} | "
                f"RMSE={round(rmse, 5)} | "
                f"RMSE_base={round(rmse_baseline, 5)} | "
                f"RMSE_0={round(rmse_zeros, 5)} | "
                f"MAE={round(mae, 5)} | "
                f"MAE_base={round(mae_baseline, 5)} | "
                f"MAE_0={round(mae_zeros, 5)} | "
                f"Params={params_optimos if params_optimos else f'Alpha={alpha_optimo}'}"
            )

        fecha_fin = marcar_bloque_como_done(worksheet, filas_bloque, resultados_metricas)

        print("FIN DE BLOQUE")
        print(f"Fecha inicio: {fecha_ini}")
        print(f"Fecha fin   : {fecha_fin}")
        print(f"Targets ejecutados: {len(resultados_bloque)}")

        return resultados_bloque, info_split

    except Exception as e:
        marcar_bloque_como_error(worksheet, filas_bloque)
        print("ERROR en el bloque:")
        print(str(e))
        raise

In [12]:
FECHA_TRAIN_FIN = "2024-W29"
FECHA_TEST_INI = "2024-W30"

## Ejecución del experimento

El bucle llama a `run_block_daily` para cada método hasta agotar todos los bloques pendientes del sheet.

In [15]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings(
    "ignore",
    message="`sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel`*",
    category=UserWarning
)
# metodos = ["LinearRegression", "Ridge", "RandomForestRegressor", "GradientBoostingRegressor"]
metodos = ["Ridge", "RandomForestRegressor", "GradientBoostingRegressor"]
for metodo in metodos:
    while True:
        resultado = run_block_daily(metodo)
        if resultado is None:
            # No quedan más bloques pendientes para este método
            break

No quedan bloques diarios pendientes para Ridge.
No quedan bloques diarios pendientes para RandomForestRegressor.
INICIO DE BLOQUE
Archivo: df_diario_4.csv
Método: GradientBoostingRegressor
Buscando hiperparámetros óptimos GBR para este archivo...
  Mejores parámetros GBR: {'subsample': 0.8, 'n_estimators': 100, 'min_samples_leaf': 30, 'max_features': 'log2', 'max_depth': 2, 'learning_rate': 0.01}
Parámetros fijados para 'df_diario_4.csv': {'subsample': 0.8, 'n_estimators': 100, 'min_samples_leaf': 30, 'max_features': 'log2', 'max_depth': 2, 'learning_rate': 0.01}
INFORMACIÓN DEL SPLIT TEMPORAL DEL BLOQUE
Total observaciones: 1270
Train: 1000 filas
Test : 270 filas
Train fechas: 2021-01-11 -> 2024-12-31
Test  fechas: 2025-01-02  -> 2026-01-30
Target: target_AGG | RMSE=0.00282 | RMSE_base=0.00386 | RMSE_0=0.00281 | MAE=0.00219 | MAE_base=0.00305 | MAE_0=0.00219 | Params={'subsample': 0.8, 'n_estimators': 100, 'min_samples_leaf': 30, 'max_features': 'log2', 'max_depth': 2, 'learning_rate

Para el método Ridge: Los archivos df_semanal_1, 2, 3 y 4 dan resultados prácticamente idénticos entre sí y muy parecidos al df_semanal original. Eso indica que los lags adicionales que añadiste en esos archivos no están aportando señal nueva con Ridge. Random Forest puede capturar esas interacciones mejor.

In [14]:
# df = pd.read_csv("../../Datos_csv/df_semanal_2.csv")
# pd.set_option("display.max_columns", None) 
# df.head()